# SmolVLM-256M: EVLM Benchmark

This notebook runs the benchmark of the EVLM paper (Farahani et al., *EVLM: Intent-Driven Edge Vision Language Model for UAV-Based Power Line Inspection*, EDGE 2026) with the fastest inference setup of this repository.

The paper uses two public datasets (Section V-B):

- **Power line videos** (Xing et al., IROS 2023, ref. [35]): 720x480 frames at 30 fps from eight environments. The paper cuts them into 4 s clips of 120 frames. This dataset is used for the speed tests.
- **InsPLAD** (Vieira e Silva et al., 2023, ref. [43]): UAV images of power line parts with labels. We use the fault classification crops (good, rust and other defects). This dataset is used for the accuracy test.

`get_data.py` downloads only the parts that the test needs: one 4 s clip for each environment (about 0.5 GB) and the InsPLAD fault crops (about 0.9 GB). The full archives are 25 GB and 6.4 GB.

`evlm_eval.py` has two tasks:

1. `clips`: for each clip, a histogram filter keeps the key frames (512 bins, threshold 0.95, as in the paper). The VLM answers the operator intent for each key frame. The script prints the key frames, the filter time, the VLM time and the segment latency.
2. `insplad`: the VLM answers the two prompts of the paper's Table III for each image. The script scores the component name and the rust answer against the labels.

The defaults are the best setup from the earlier tests: fast GPU preprocessing, `--size 1024`, CUDA-graph decoding, stop at EOS, and a TensorRT engine for SigLIP. Each part has a flag, so the last cell can turn the parts off one at a time (ablation).

We use the base model. The paper fine-tunes it with LoRA, and its weights are not public, so the accuracy here is lower than in the paper.

In [ ]:
REPO = "./edge-ai-smol-lvm-250-optimization"
DATA = f"{REPO}/data"
OUT = f"{REPO}/results"

# Colab: packages for the TensorRT engine. The Jetson has TensorRT in JetPack.
!pip install -q tensorrt onnx

# dataset setup: about 1.4 GB, only the files the test needs; files that exist already are skipped
!python {REPO}/get_data.py --out {DATA}

In [ ]:
# speed test on the power line clips, best setup
!python {REPO}/evlm_eval.py --task clips --data {DATA} --out {OUT}

In [ ]:
# accuracy test on InsPLAD (10 images for each asset and label), best setup
!python {REPO}/evlm_eval.py --task insplad --data {DATA} --out {OUT} --limit 10

In [ ]:
# ablation: turn off one part at a time (two environments, to keep the run short)
import json
import subprocess

ENVS = "forest,sand"
CONFIGS = {
    "best": [],
    "no TensorRT": ["--no-trt"],
    "default tiles": ["--size", "0"],
    "no stop at EOS": ["--no-stop"],
    "eager decode": ["--decode", "eager"],
    "bare generate": ["--decode", "bare"],
    "slow preprocess": ["--preprocess", "slow"],
    "no frame filter": ["--filter", "none"],
    "nothing (baseline)": ["--preprocess", "slow", "--size", "0", "--decode", "bare", "--no-trt"],
}

print(f"{'config':<20} {'key frames':>10} {'VLM ms/key':>10} {'segment s':>9}")
for name, flags in CONFIGS.items():
    r = subprocess.run(
        ["python", f"{REPO}/evlm_eval.py", "--task", "clips", "--data", DATA, "--out", OUT, "--envs", ENVS, *flags],
        capture_output=True, text=True,
    )
    line = next((l for l in r.stdout.splitlines() if l.startswith("SUMMARY")), None)
    if line is None:
        print(f"{name:<20} failed: {(r.stderr.strip().splitlines() or ['?'])[-1]}")
        continue
    s = json.loads(line[len("SUMMARY "):])
    print(f"{name:<20} {s['key_frames']:10.2f} {s['vlm_ms_per_key']:10.1f} {s['segment_s']:9.2f}")